## 1. Imports

In [ ]:
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

CSV_PATH = "Food_Inspections_20240215.csv"


## 2. Load Data

In [ ]:
df = pd.read_csv(CSV_PATH, encoding="utf-8", dtype=str)
df.columns = [c.strip() for c in df.columns]
for col in df.columns:
    df[col] = df[col].str.strip()

RAW_COLUMNS = list(df.columns)
num_rows = len(df)
df.shape


## 3. Derived Columns

In [ ]:
df["inspection_date_parsed"] = pd.to_datetime(df["Inspection Date"], errors="coerce")


def count_violations(val):
    if pd.isna(val) or str(val).strip() == "":
        return 0
    return len([s for s in str(val).split("|") if s.strip()])


df["No. of Violations"] = df["Violations"].apply(count_violations).astype(int)

for col in ["Inspection ID", "License #", "Latitude", "Longitude"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")


## 4. Helper Functions

In [ ]:
def safe_str_lengths(series):
    return series.dropna().astype(str).str.len()


def infer_type(col, series):
    if col == "Inspection Date":
        return "datetime"
    numeric_series = pd.to_numeric(series.dropna(), errors="coerce")
    if numeric_series.notna().sum() / max(series.notna().sum(), 1) >= 0.8:
        return "numeric"
    return "text"


def first_nonzero_digit(val):
    try:
        s = str(int(abs(float(val))))
        for ch in s:
            if ch != "0":
                return int(ch)
    except Exception:
        pass
    return None


def iqr_outliers(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lb, ub = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return lb, ub


## 5. Cardinality Profile

In [ ]:
profile_rows = []

for col in RAW_COLUMNS:
    series = df[col]
    null_count = series.isna().sum()
    distinct = series.nunique(dropna=True)
    inferred = infer_type(col, series)

    str_min = str_max = str_med = str_mean = np.nan
    if inferred in ("text", "datetime"):
        lens = safe_str_lengths(series)
        if len(lens):
            str_min, str_max = lens.min(), lens.max()
            str_med, str_mean = lens.median(), lens.mean()

    vc = series.value_counts(dropna=False)
    constancy = vc.iloc[0] / num_rows if len(vc) else np.nan

    profile_rows.append({
        "column": col, "num_rows": num_rows,
        "null_count": null_count, "null_pct": round(null_count / num_rows * 100, 2),
        "distinct": distinct, "uniqueness": round(distinct / num_rows, 4),
        "str_len_min": str_min, "str_len_max": str_max,
        "str_len_median": str_med,
        "str_len_mean": round(str_mean, 2) if not np.isnan(str_mean) else np.nan,
        "constancy": round(constancy, 4), "inferred_type": inferred,
    })

cardinality_profile = pd.DataFrame(profile_rows)
cardinality_profile


## 6. Numeric Distributions

In [ ]:
NUMERIC_COLS = ["Inspection ID", "License #", "Latitude", "Longitude", "No. of Violations"]

numeric_stats = []
for col in NUMERIC_COLS:
    s = pd.to_numeric(df[col], errors="coerce").dropna()
    if s.empty:
        continue
    numeric_stats.append({
        "column": col,
        "min": s.min(), "max": s.max(),
        "mean": round(s.mean(), 4), "median": s.median(),
        "variance": round(s.var(), 4),
        "Q1": s.quantile(0.25), "Q2": s.quantile(0.50), "Q3": s.quantile(0.75),
    })
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(s, bins=30, color="steelblue", edgecolor="white")
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()

pd.DataFrame(numeric_stats)


## 7. Categorical Value Counts (Top 20)

In [ ]:
CATEGORICAL_COLS = ["Facility Type", "Risk", "City", "State", "Inspection Type", "Results"]
BAR_PLOT_COLS = ["Facility Type", "Risk", "Inspection Type", "Results", "City"]

for col in CATEGORICAL_COLS:
    vc = df[col].value_counts(dropna=False).head(20)
    vc_norm = df[col].value_counts(normalize=True, dropna=False).head(20).round(4)
    display(pd.DataFrame({"count": vc, "freq": vc_norm}))

    if col in BAR_PLOT_COLS:
        fig, ax = plt.subplots(figsize=(10, 5))
        vc.plot(kind="bar", ax=ax, color="teal", edgecolor="white")
        ax.set_title(col)
        ax.set_ylabel("Count")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()


## 8. Violation Code Frequency

In [ ]:
viol_codes = (
    df["Violations"].dropna()
    .str.split("|", expand=False)
    .explode()
    .str.strip()
    .str.extract(r"^(\d+)\.")
    [0]
    .dropna()
    .astype(int)
)

viol_freq = viol_codes.value_counts().sort_index()
viol_freq.name = "count"
display(viol_freq.to_frame())

fig, ax = plt.subplots(figsize=(14, 5))
viol_freq.plot(kind="bar", ax=ax, color="teal", edgecolor="white")
ax.set_title("Violation Code Frequency")
ax.set_xlabel("Violation Code")
ax.set_ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 9. Type & Pattern Analysis

In [ ]:
# Zip
zip_str = df["Zip"].fillna("").astype(str).str.strip()
zip_lengths = zip_str.str.len()
display(zip_lengths.value_counts().sort_index())
print(f"5-char: {(zip_lengths == 5).mean():.4f}  |  "
      f"digits-only: {zip_str.str.fullmatch(r'\d+').fillna(False).mean():.4f}")

# State
state_vals = df["State"].dropna().unique()
frac_il = (df["State"].str.upper().str.strip() == "IL").mean()
print(f"States: {sorted(state_vals)}  |  IL fraction: {frac_il:.4f}")

# City
display(df["City"].value_counts(dropna=False).head(10))
city_upper = df["City"].str.upper()
print(f"CHICAGO exact: {(df['City'] == 'CHICAGO').sum()}  |  "
      f"Chicago title: {(df['City'] == 'Chicago').sum()}  |  "
      f"any variant: {city_upper.str.strip().eq('CHICAGO').sum()}")

# Location
loc_pattern = re.compile(r"^\s*\(\s*-?\d+\.?\d*\s*,\s*-?\d+\.?\d*\s*\)\s*$")
loc_valid = df["Location"].dropna().apply(lambda x: bool(loc_pattern.match(x)))
print(f"Location pattern match: {loc_valid.sum() / len(df):.4f}")


## 10. Validity Profile

In [ ]:
def validity_row(check_name, valid_mask):
    v = valid_mask.sum()
    inv = (~valid_mask).sum()
    return {"check": check_name, "valid": v, "valid_pct": round(v / num_rows * 100, 2),
            "invalid": inv, "invalid_pct": round(inv / num_rows * 100, 2)}


validity_profile = pd.DataFrame([
    validity_row("Inspection Date parseable", df["inspection_date_parsed"].notna()),
    validity_row("Zip 5 digits",
                 df["Zip"].fillna("").astype(str).str.strip().str.fullmatch(r"\d{5}").fillna(False)),
    validity_row("State == IL", df["State"].str.upper().str.strip().eq("IL").fillna(False)),
    validity_row("Lat in [41.5, 42.2]",
                 pd.to_numeric(df["Latitude"], errors="coerce").pipe(
                     lambda s: s.notna() & s.between(41.5, 42.2))),
    validity_row("Lon in [-88, -87]",
                 pd.to_numeric(df["Longitude"], errors="coerce").pipe(
                     lambda s: s.notna() & s.between(-88.0, -87.0))),
])
validity_profile


## 11. Outlier Detection (IQR) — Coordinates

In [ ]:
outlier_summary = []
for col in ["Latitude", "Longitude"]:
    s = pd.to_numeric(df[col], errors="coerce").dropna()
    if s.empty:
        continue
    lb, ub = iqr_outliers(s)
    n_out = pd.to_numeric(df[col], errors="coerce").apply(
        lambda x: pd.notna(x) and (x < lb or x > ub)).sum()
    outlier_summary.append({"column": col, "lower": round(lb, 4), "upper": round(ub, 4),
                            "outliers": n_out, "pct": round(n_out / num_rows * 100, 2)})

outlier_df = pd.DataFrame(outlier_summary)
outlier_df


## 12. Rare Categories (< 1%)

In [ ]:
for col in ["Facility Type", "Inspection Type", "Results"]:
    vc = df[col].value_counts(normalize=True)
    rare = vc[vc < 0.01]
    if not rare.empty:
        display(rare)


## 13. Benford's Law — License # and Inspection ID

In [ ]:
benford_ref = {d: np.log10(1 + 1 / d) for d in range(1, 10)}

for col in ["License #", "Inspection ID"]:
    series = pd.to_numeric(df[col], errors="coerce").dropna()
    if series.empty:
        continue

    digits = series.apply(first_nonzero_digit).dropna().astype(int)
    digit_counts = digits.value_counts().reindex(range(1, 10), fill_value=0).sort_index()
    digit_freq = digit_counts / digit_counts.sum()

    display(pd.DataFrame({
        "digit": range(1, 10), "count": digit_counts.values,
        "frequency": digit_freq.values.round(4),
        "benford_expected": [round(benford_ref[d], 4) for d in range(1, 10)],
    }))

    fig, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(1, 10)
    ax.bar(x - 0.2, digit_freq.values, width=0.4, label="Empirical", color="steelblue")
    ax.bar(x + 0.2, [benford_ref[d] for d in range(1, 10)], width=0.4,
           label="Benford", color="salmon", alpha=0.8)
    ax.set_xticks(x)
    ax.set_title(col)
    ax.set_xlabel("First Digit")
    ax.set_ylabel("Frequency")
    ax.legend()
    plt.tight_layout()
    plt.show()
